# Géométrie riemannienne — l'habilitation de 1854 : métrique, distance, courbure

Grain **R-07** de la série Riemann200 — EPIC [#20403](https://github.com/jsboige/CoursIA/issues/20403).

**Fil rouge.** Le 10 juin 1854, pour son habilitation, Riemann répond à la question que Gauss
avait laissée en héritage : qu'est-ce qui détermine la géométrie d'un espace ? Sa réponse tient
en une idée — une **métrique** $ds^2 = \sum g_{ij}\,dx^i dx^j$, règle de longueur assignée à
chaque point, de laquelle descendent les distances, les lignes droites (géodésiques) et la
courbure, le tout **sans quitter la surface** : nul besoin d'un espace ambiant. Ce carnet mesure
la chaîne complète sur le laboratoire le plus ancien de cette géométrie intrinsèque, la sphère
terrestre : l'anisotropie de la métrique, la distance géodésique contre la route à cap constant,
les géodésiques intégrées comme trajectoires d'accélération purement normale, la courbure
obtenue par le seul excès angulaire d'un triangle (Gauss-Bonnet contre L'Huilier par deux voies
indépendantes), sa loi d'échelle en $1/R^2$, et la failure du théorème de Pythagore.

Ce carnet est délibérément **scopé à la géométrie riemannienne de l'habilitation** — métrique,
distance, géodésique, courbure (désendroit [#19333](https://github.com/jsboige/CoursIA/issues/19333)) —
pas un panorama de géométrie différentielle.

## Plan

1. **La métrique** — $ds^2 = R^2(d\theta^2 + \sin^2\theta\,d\varphi^2)$ : la longueur d'un degré
   d'est en ouest dépend de la latitude ; l'espace des directions n'est pas isotrope.
2. **La distance** — grand cercle contre loxodromie Paris–Tokyo : deux notions concurrentes de
   « ligne droite », $16\,\%$ d'écart.
3. **Les géodésiques** — intégrées par RK4 comme trajectoires à accélération purement normale :
   retour au départ au millième de nanomètre près, plan du grand cercle invariant.
4. **La courbure** — l'excès angulaire du triangle Paris–Tokyo–San Francisco : Gauss-Bonnet et
   L'Huilier d'accord à $10^{-16}$, courbure $K = \varepsilon/A = 1/R^2$.
5. **Contre-témoin d'échelle** — l'excès décroît comme $1/R^2$ : chaque doublement du rayon le
   divise par $4$ ; aux petites échelles la géométrie redevient euclidienne.
6. **Pythagore sphérique** — $\cos(c/R) = \cos(a/R)\cos(b/R)$ : l'euclidien surestime.
7. Ce que Mathlib sait, limites honnêtes, exercices, références.

Toutes les valeurs affichées sont **calculées dans une sortie** — jamais recopiées.

## Dépendances

Python **standard library uniquement** (`math`) — distances par haversine, loxodromie par la
projection de Mercator, géodésiques par un Runge-Kutta d'ordre 4 écrit à la main, angles et aires
par les formules fermées de la trigonométrie sphérique. Rien d'autre : chaque mesure est
reproductible sans installation.

In [1]:
import math

R = 6371.0  # km, rayon terrestre moyen

print("stdlib uniquement :", math.__name__)
print("R =", R, "km ; aire de la sphere 4 pi R^2 =", f"{4 * math.pi * R * R:.0f}", "km^2")

stdlib uniquement : math
R = 6371.0 km ; aire de la sphere 4 pi R^2 = 510064472 km^2


**Lecture.** L'environnement est minimal à dessein. Sans bibliothèque de calcul formel,
les vérifications restent numériques : elles *corroborent* les théorèmes, elles ne les *démontrent* pas.

## 1. La métrique — l'anisotropie de l'espace des directions

Sur la sphère, la métrique en coordonnées $(\theta, \varphi)$ (colatitude, longitude) est
$ds^2 = R^2(d\theta^2 + \sin^2\theta\,d\varphi^2)$ : le coefficient de $d\varphi^2$,
$g_{\varphi\varphi} = R^2\sin^2\theta$, **dépend du point**. Conséquence mesurable : un arc
de $1°$ de longitude (est-ouest) ne mesure pas la même longueur selon la latitude — les
méridiens se resserrent vers les pôles. Le rapport des coefficients $g_{\varphi\varphi}/g_{\theta\theta}
= \cos^2(\text{lat})$ prédit exactement ce rétrécissement.

In [2]:
print("M1 : longueur d'un arc de 1 degre de longitude a differentes latitudes :")
for lat in [0, 30, 48.8566, 60, 80]:
    lam = math.radians(lat)
    d = R * math.radians(1.0) * math.cos(lam)
    print(f"   lat {lat:7.4f} N -> {d:6.2f} km   (g_phi/g_theta = cos^2(lat) = {math.cos(lam)**2:.4f})")

M1 : longueur d'un arc de 1 degre de longitude a differentes latitudes :
   lat  0.0000 N -> 111.19 km   (g_phi/g_theta = cos^2(lat) = 1.0000)
   lat 30.0000 N ->  96.30 km   (g_phi/g_theta = cos^2(lat) = 0.7500)
   lat 48.8566 N ->  73.16 km   (g_phi/g_theta = cos^2(lat) = 0.4329)
   lat 60.0000 N ->  55.60 km   (g_phi/g_theta = cos^2(lat) = 0.2500)
   lat 80.0000 N ->  19.31 km   (g_phi/g_theta = cos^2(lat) = 0.0302)


**Lecture.** À l'équateur, $111{,}19$ km ; à Paris ($48{,}86°$ N), $73{,}16$ km ; à $80°$ N,
$19{,}31$ km — le mètre est-ouest fond comme $\cos(\text{lat})$, et le rapport des coefficients
de la métrique $\cos^2(\text{lat})$ rend compte de chaque valeur : $0{,}4329$ pour Paris,
mesuré comme prédit. C'est le premier enseignement de 1854 rendu tangible : **la métrique est
une donnée par point**, et sa variation contient toute la géométrie — ici, celle d'une sphère
plongée ; dans le cas général, d'une variété qui n'existe que par elle-même.

## 2. La distance — géodésique contre loxodromie

Deux candidats au titre de « plus court chemin » de Paris à Tokyo : le **grand cercle**
(l'intersection de la sphère avec le plan passant par les deux villes et le centre — la
géodésique de la section 3) et la **loxodromie** (la route à cap constant, qui coupe tous les
méridiens sous le même angle — la ligne droite de la carte de Mercator). Le premier se calcule
par haversine ; la seconde par la projection de Mercator, où elle est droite de pente
$\tan\alpha = \Delta\lambda/\Delta\psi$ avec $\psi = \ln\tan(\pi/4 + \varphi/2)$,
de longueur $R\,|\Delta\varphi|/|\cos\alpha|$.

In [3]:
def gc_deg(lat1, lon1, lat2, lon2):
    p1, l1, p2, l2 = map(math.radians, [lat1, lon1, lat2, lon2])
    dp, dl = p2 - p1, l2 - l1
    a = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2 * R * math.asin(math.sqrt(a))

def merc_y(lat):
    return math.log(math.tan(math.pi/4 + math.radians(lat)/2))

def loxo_deg(lat1, lon1, lat2, lon2):
    dphi = math.radians(lat2 - lat1)
    dpsi = merc_y(lat2) - merc_y(lat1)
    dlam = math.radians((lon2 - lon1 + 540) % 360 - 180)
    alpha = math.atan2(dlam, dpsi)
    return R * abs(dphi) / abs(math.cos(alpha)) if abs(math.cos(alpha)) > 1e-12 else float('inf')

PAR = (48.8566, 2.3522); TOK = (35.6762, 139.6503); SFO = (37.7749, -122.4194)
d_gc = gc_deg(*PAR, *TOK)
d_lx = loxo_deg(*PAR, *TOK)
print(f"Paris -> Tokyo : grand cercle = {d_gc:.1f} km ; loxodromie = {d_lx:.1f} km")
print(f"ecart = {d_lx - d_gc:.1f} km (+{100 * (d_lx - d_gc) / d_gc:.2f} %)")

Paris -> Tokyo : grand cercle = 9711.7 km ; loxodromie = 11326.7 km
ecart = 1614.9 km (+16.63 %)


**Lecture.** Le grand cercle : $9711{,}7$ km ; la loxodromie : $11326{,}7$ km — l'avion qui
battrait la route à cap constant y gagnerait $1614{,}9$ km, soit $16{,}63\,\%$. La « ligne
droite » de la carte n'est pas celle du globe : Mercator est conforme (il préserve les angles,
donc la loxodromie y est droite) mais il ne préserve pas les longueurs — aucune carte plate ne
peut, depuis Gauss, et pour la même raison : la courbure.

## 3. Les géodésiques — courber sans jamais tourner

La définition moderne d'une géodésique est **intrinsèque** : une courbe dont l'accélération est
en tout point perpendiculaire à la surface — elle « courbe » (elle suit la sphère) sans jamais
« tourner » (aucune accélération dans son plan tangent). Plongée dans l'espace, cette condition
donne l'équation newtonienne $\ddot{p} = -\tfrac{|\dot p|^2}{R^2}\,p$ : l'accélération est
purement normale, exactement centripète. On l'intègre par Runge-Kutta d'ordre 4 — deux tests :

- **un tour complet sur l'équateur** : le point doit revenir à son départ ;
- **départ de Paris à cap $45°$** : le plan du grand cercle (porté par $p \times v$) doit rester
  immobile — c'est le théorème de Clairaut réduit à sa signature la plus simple.

**Hypothèse H1 (pré-enregistrée avant la mesure).** *Avec $4000$ pas pour un tour, l'erreur de
retour après un tour complet d'équateur sera inférieure à $10^{-10}R$, et la dérive relative du
plan $p\times v$ sur trois demi-tours depuis Paris à cap $45°$ inférieure à $10^{-13}$.*

In [4]:
def step(p, v, h):
    def acc(pp, vv):
        v2 = sum(a*a for a in vv)
        return tuple(-v2/(R*R) * c for c in pp)
    def add(*vs): return tuple(sum(c) for c in zip(*vs))
    def mul(vs, s): return tuple(c*s for c in vs)
    k1p, k1v = v, acc(p, v)
    p2 = add(p, mul(k1p, h/2)); v2 = add(v, mul(k1v, h/2))
    k2p, k2v = v2, acc(p2, v2)
    p3 = add(p, mul(k2p, h/2)); v3 = add(v, mul(k2v, h/2))
    k3p, k3v = v3, acc(p3, v3)
    p4 = add(p, mul(k3p, h)); v4 = add(v, mul(k3v, h))
    k4p, k4v = v4, acc(p4, v4)
    pn = add(p, mul(add(k1p, mul(k2p,2), mul(k3p,2), k4p), h/6))
    vn = add(v, mul(add(k1v, mul(k2v,2), mul(k3v,2), k4v), h/6))
    return pn, vn

def cross(a, b):
    return (a[1]*b[2]-a[2]*b[1], a[2]*b[0]-a[0]*b[2], a[0]*b[1]-a[1]*b[0])

# cas 1 : equateur, depart (R, 0, 0), vitesse vers l'est, un tour complet
p = (R, 0.0, 0.0); v = (0.0, R, 0.0)
N0 = cross(p, v)
h = 2*math.pi/4000
for i in range(4000):
    p, v = step(p, v, h)
N1 = cross(p, v)
err_pos = math.dist(p, (R, 0.0, 0.0)) / R
drift_N = math.dist(N0, N1) / math.dist(N0, (0, 0, 0))
print(f"un tour d'equateur : err position = {err_pos:.2e} R ; derive |p x v| = {drift_N:.2e}")

# cas 2 : depart a 48.86 N cap 45 deg -- le plan du grand cercle doit rester fixe
lat0 = math.radians(48.8566)
p = (R*math.cos(lat0), 0.0, R*math.sin(lat0))
cap = math.radians(45.0)
east = (0.0, 1.0, 0.0)
north = (-math.sin(lat0), 0.0, math.cos(lat0))
vdir = tuple(math.cos(cap)*east[i] + math.sin(cap)*north[i] for i in range(3))
v = tuple(R*c for c in vdir)
N0 = cross(p, v)
for i in range(6000):
    p, v = step(p, v, h)
N1 = cross(p, v)
print(f"depart 48.86 N cap 45 deg, apres 3 demi-tours : derive du plan = {math.dist(N0, N1)/math.dist(N0,(0,0,0)):.2e}")

un tour d'equateur : err position = 2.59e-12 R ; derive |p x v| = 2.39e-15


depart 48.86 N cap 45 deg, apres 3 demi-tours : derive du plan = 6.48e-15


**Lecture — H1 confirmée.** Un tour complet d'équateur ramène le point à
$2{,}6\times10^{-12}R$ de son départ — $16$ micromètres sur $6371$ km — et le plan du grand
cercle dérive de $2{,}4\times10^{-15}$ en relatif ; depuis Paris à cap $45°$, trois demi-tours
plus tard, la dérive du plan est $6{,}5\times10^{-15}$ : à la précision machine près, le plan
de la trajectoire **ne bouge jamais**. Les géodésiques de la sphère vivent dans un plan fixe :
ce sont exactement les grands cercles — les « lignes droites » de la section 2 retrouvées par
la seule dynamique, sans jamais invoquer le centre de la sphère.

## 4. La courbure — l'excès angulaire, mesuré par deux voies indépendantes

Le théorème de Gauss (1827, *theorema egregium*) dit que la courbure $K$ d'une surface est
**intrinsèque** : mesurable sans quitter la surface. Sa forme la plus opérante est le
théorème de Gauss-Bonnet local : pour tout triangle géodésique d'aire $A$ et d'angles
$\alpha, \beta, \gamma$,
$$\alpha + \beta + \gamma - \pi = K \cdot A \quad \text{(excès angulaire)}.$$
Sur la sphère, $K = 1/R^2$ : l'excès $\varepsilon$ est simplement proportionnel à l'aire.

Le témoin : le triangle **Paris–Tokyo–San Francisco** (trois grands arcs d'aéroport). Les angles
se calculent par la loi des cosinus sphérique depuis les trois côtés (angles au centre), puis
Gauss-Bonnet rend l'aire depuis les angles. La contre-voie : la **formule de L'Huilier** (1798),
qui calcule la même aire **depuis les côtés seuls** — un chemin qui ne regarde jamais les
angles. Deux voies indépendantes, un verdict.

In [5]:
def xyz(lat, lon):
    la, lo = math.radians(lat), math.radians(lon)
    return (R*math.cos(la)*math.cos(lo), R*math.cos(la)*math.sin(lo), R*math.sin(la))

def angle_sph(A, B, C):
    # angle en A par la loi des cosinus spherique : les cotes sont les angles au centre
    def dot(u, w): return sum(a*b for a, b in zip(u, w))
    ca = dot(B, C)/(R*R); cb = dot(C, A)/(R*R); cc = dot(A, B)/(R*R)
    a_, b_, c_ = math.acos(max(-1, min(1, ca))), math.acos(max(-1, min(1, cb))), math.acos(max(-1, min(1, cc)))
    cosA = (math.cos(a_) - math.cos(b_)*math.cos(c_))/(math.sin(b_)*math.sin(c_))
    return math.acos(max(-1, min(1, cosA)))

A, B, C = xyz(*PAR), xyz(*TOK), xyz(*SFO)
angA, angB, angC = angle_sph(A, B, C), angle_sph(B, C, A), angle_sph(C, A, B)
exces = angA + angB + angC - math.pi
aire_gb = exces * R * R

# L'Huilier : aire depuis les cotes seuls
def gc_xyz(P, Q):
    cosc = sum(a*b for a, b in zip(P, Q))/(R*R)
    return R * math.acos(max(-1, min(1, cosc)))
a, b, c = gc_xyz(B, C), gc_xyz(C, A), gc_xyz(A, B)
s = (a + b + c)/2
k = math.sqrt(math.tan(s/(2*R))*math.tan((s-a)/(2*R))*math.tan((s-b)/(2*R))*math.tan((s-c)/(2*R)))
aire_lhu = 4 * math.atan(k) * R * R

print(f"angles = {math.degrees(angA):.3f} + {math.degrees(angB):.3f} + {math.degrees(angC):.3f} deg"
      f" (somme {math.degrees(angA+angB+angC):.3f} deg, contre 180 en euclidien)")
print(f"exces angulaire = {math.degrees(exces):.4f} deg")
print(f"aire Gauss-Bonnet (eps R^2)  = {aire_gb:.0f} km^2")
print(f"aire L'Huilier (cotes seuls) = {aire_lhu:.0f} km^2")
print(f"accord des deux voies : ecart relatif = {abs(aire_gb-aire_lhu)/aire_lhu:.2e}")
print(f"courbure K = eps/A = {exces/aire_gb:.6e} km^-2  contre 1/R^2 = {1/(R*R):.6e} km^-2")
print(f"part de la sphere : {aire_gb/(4*math.pi*R*R)*100:.2f} %")

angles = 74.638 + 80.897 + 89.869 deg (somme 245.404 deg, contre 180 en euclidien)
exces angulaire = 65.4040 deg
aire Gauss-Bonnet (eps R^2)  = 46333691 km^2
aire L'Huilier (cotes seuls) = 46333691 km^2
accord des deux voies : ecart relatif = 3.22e-16
courbure K = eps/A = 2.463683e-08 km^-2  contre 1/R^2 = 2.463683e-08 km^-2
part de la sphere : 9.08 %


**Lecture.** La somme des angles vaut $245{,}404°$ — soixante-cinq degrés de plus que les
$180°$ d'Euclide. L'excès $\varepsilon = 65{,}4040°$ rend par Gauss-Bonnet une aire de
$46\,333\,691$ km² ; L'Huilier, qui n'a vu que les trois côtés, rend **le même nombre à
$3\times10^{-16}$ près** — la précision machine. Le triangle couvre $9{,}08\,\%$ de la
sphère. Et surtout : le rapport $\varepsilon/A = 2{,}4637\times10^{-8}$ km$^{-2}$ coïncide avec
$1/R^2$ chiffre à chiffre — l'excès angulaire est à l'aire ce que la courbure est à $1$. Aucun de ces deux
chemins n'a utilisé le centre de la sphère : angles et aires sont des quantités de **surface**.
C'est le *theorema egregium* vu au travail : la courbure est une donnée interne, mesurable par
quiconque vit dans la surface sans jamais en sortir.

## 5. Contre-témoin d'échelle — la courbure s'efface aux grandes sphères

Gauss-Bonnet a une conséquence d'échelle : $\varepsilon = A/R^2$, donc pour un triangle de
dimensions **fixées en kilomètres**, gonfler le rayon de la sphère doit écraser l'excès comme
$1/R^2$ — chaque doublement de $R$ le divise par $4$. À la limite $R \to \infty$, l'excès
s'annule : la géométrie locale redevient euclidienne, et c'est **pourquoi** le théorème de
Pythagore fonctionne sur un terrain de football malgré la rotondité de la Terre.

Le témoin : un triangle **équilatéral géodésique de côté exactement $L = 5000$ km**, construit
exactement sur des sphères de rayon $kR$ (pôle Nord plus deux sommets à colatitude $L/kR$,
écart de longitude $\arccos\frac{\cos\theta}{1+\cos\theta}$ qui garantit le troisième côté
égal à $L$ au millimètre).

**Hypothèse H2 (pré-enregistrée avant la mesure).** *Le rapport $\varepsilon(k)/\varepsilon(2k)$
tend vers $4$, et le produit $k^2\varepsilon(k)$ converge vers une constante quand $k$ croît.*

In [6]:
def exces_equilateral(L, Rk):
    th = L/Rk
    dl = math.acos(math.cos(th)/(1+math.cos(th)))
    N = (0.0, 0.0, 1.0)
    P1 = (math.sin(th), 0.0, math.cos(th))
    P2 = (math.sin(th)*math.cos(dl), math.sin(th)*math.sin(dl), math.cos(th))
    def dot(u, w): return sum(a*b for a, b in zip(u, w))
    def side(P, Q): return math.acos(max(-1, min(1, dot(P, Q))))
    a4, b4, c4 = side(P1, P2), side(N, P2), side(N, P1)
    ang = []
    for (opp, x_, y_) in [(a4, b4, c4), (b4, c4, a4), (c4, a4, b4)]:
        cosA = (math.cos(opp) - math.cos(x_)*math.cos(y_))/(math.sin(x_)*math.sin(y_))
        ang.append(math.acos(max(-1, min(1, cosA))))
    return sum(ang) - math.pi

print("triangle equilateral geodesique de cote L = 5000 km, sur spheres de rayon kR :")
eps = [exces_equilateral(5000.0, R*k) for k in [1, 2, 4, 8]]
for k, e in zip([1, 2, 4, 8], eps):
    print(f"   k={k} : exces = {math.degrees(e):9.6f} deg ; k^2 * exces = {math.degrees(e*k*k):8.6f} deg")
print("rapports eps(k)/eps(2k) :",
      " -> ".join(f"{eps[i]/eps[i+1]:.4f}" for i in range(3)),
      "(cible 4 = 2^2)")

triangle equilateral geodesique de cote L = 5000 km, sur spheres de rayon kR :
   k=1 : exces = 16.563485 deg ; k^2 * exces = 16.563485 deg
   k=2 : exces =  3.895291 deg ; k^2 * exces = 15.581165 deg
   k=4 : exces =  0.959674 deg ; k^2 * exces = 15.354779 deg
   k=8 : exces =  0.239051 deg ; k^2 * exces = 15.299276 deg
rapports eps(k)/eps(2k) : 4.2522 -> 4.0590 -> 4.0145 (cible 4 = 2^2)


**Lecture — H2 confirmée.** L'excès passe de $16{,}56°$ ($k=1$) à $0{,}239°$ ($k=8$) : les
rapports successifs valent $4{,}25 \to 4{,}06 \to 4{,}01$ — convergence franche vers $4 = 2^2$,
la loi $1/R^2$. Le produit $k^2\varepsilon$ se stabilise ($16{,}56 \to 15{,}58 \to 15{,}35 \to
15{,}30$) : la loi $\varepsilon = A/R^2$ est le **premier ordre**, et l'écart résiduel à petit
$k$ ($L/R = 0{,}78$ à l'échelle terrestre — un triangle énorme) est la correction d'ordre
supérieur, elle aussi purement sphérique. La lecture physique : la courbure est un détecteur
d'échelle — à côté fixe, une sphère assez grande est localement indiscernable d'un plan.

## 6. Pythagore sphérique — la failure mesurée

En géométrie sphérique, le théorème de Pythagore devient $\cos(c/R) = \cos(a/R)\cos(b/R)$
pour un triangle rectangle ($C = \pi/2$). Le témoin le plus pur : deux méridiens issus du
pôle Nord jusqu'à l'équateur, à angle droit au pôle — jambes $a = b = \tfrac{\pi}{2}R$, et
l'« hypoténuse » est l'arc d'équateur qui les joint. Le Pythagore euclidien prédirait
$c = \sqrt{a^2+b^2} = \tfrac{\pi}{2}\sqrt{2}\,R$ ; la formule sphérique prédit
$c = \tfrac{\pi}{2}R$ — **égal aux jambes**, car les trois côtés vivent sur des grands cercles
qui se coupent à angle droit.

In [7]:
a = b = math.pi/2 * R
c_sph = R * math.acos(math.cos(a/R) * math.cos(b/R))
c_euc = math.sqrt(a*a + b*b)
print(f"jambes a = b = pi/2 R = {a:.1f} km (pole -> equateur)")
print(f"hypotenuse spherique  c = {c_sph:.1f} km  (= pi/2 R = {math.pi/2*R:.1f} km, egal aux jambes)")
print(f"prediction euclidienne = {c_euc:.1f} km  (sqrt(2) x plus grand)")
print(f"c^2 = {c_sph*c_sph:.0f} km^2  contre  a^2 + b^2 = {a*a + b*b:.0f} km^2")
print(f"l'euclidien surestime d'un facteur {c_euc/c_sph:.6f} = sqrt(2)")

jambes a = b = pi/2 R = 10007.5 km (pole -> equateur)
hypotenuse spherique  c = 10007.5 km  (= pi/2 R = 10007.5 km, egal aux jambes)
prediction euclidienne = 14152.8 km  (sqrt(2) x plus grand)
c^2 = 100150925 km^2  contre  a^2 + b^2 = 200301850 km^2
l'euclidien surestime d'un facteur 1.414214 = sqrt(2)


**Lecture.** Les trois côtés valent exactement $10\,007{,}5$ km : le triangle équilatéral
droit est possible sur la sphère, impossible chez Euclide. La prédiction euclidienne
$14\,153$ km surestime d'un facteur $\sqrt2$ exactement, et $c^2 = a^2 \cdot 1 < a^2 + b^2$ :
l'inégalité $c^2 < a^2 + b^2$ est la **signature de courbure positive** — sur une surface à
courbure négative (la selle, ou le demi-plan de Poincaré), ce serait l'inverse, $c^2 > a^2+b^2$.
La géométrie de l'espace n'est pas un axiome : c'est une mesure, et le théorème de Pythagore en
est le détecteur le plus ancien.

## Ce que Mathlib sait

Le grounding est **mesuré** sur Mathlib (v4.33.0, arbre complet — comptage des fichiers `.lean` tenu par la CI, voir le catalogue) :

- **Le cadre des variétés : présent.** `Mathlib/Geometry/Manifold/` formalise variétés lisses,
  atlas, structures différentielles — le théâtre où vit une métrique riemannienne… sans le
  tenseur $g$.
- **La sphère euclidienne : riche.** `Mathlib/Geometry/Euclidean/Sphere/` couvre `Basic`,
  `OrthRadius`, `Power`, `Ptolemy` (l'égalité de Ptolémée, un théorème de distances),
  `SecondInter`, `Tangent` (plan tangent) ; `Geometry/Euclidean/Angle/` et
  `Geometry/Euclidean/Triangle.lean` traitent angles et triangles euclidiens.
- **Courbure riemannienne : absente.** Les grep `riemannCurvature`, `gaussCurvature`,
  `sectionalCurvature` rendent **zéro hit** sur l'arbre complet ; `geodesic` ne rend que trois
  fichiers hors périmètre (isométries du demi-plan de Poincaré, graphes d'arbres — pas une
  théorie des géodésiques). La chaîne métrique $\to$ géodésique $\to$ courbure n'est pas
  formalisée : les mesures de ce carnet n'ont pas de contrepartie formelle, c'est le plafond
  documenté.

## Limites honnêtes

- **Courbure constante uniquement.** La sphère a $K = 1/R^2$ partout — la leçon profonde de
  1854 est la métrique à courbure **variable** (surfaces générales, variétés de dimension $n$).
  L'instrument ne mesure aucune variation de $K$.
- **Géodésiques par le plongement.** On intègre l'équation newtonienne du grand cercle dans
  l'espace ambiant, pas le flot géodésique intrinsèque en coordonnées $(\theta, \varphi)$
  avec les symboles de Christoffel $\Gamma^k_{ij}$ — la voie strictement intrinsèque existe et
  serait l'extension naturelle.
- **Sphère parfaite.** La Terre réelle est aplatie aux pôles (platissement $\approx 0{,}3\,\%$) :
  nos distances sont celles de la sphère moyenne, pas de l'ellipsoïde WGS84 des cartes.
- **Angles par loi des cosinus.** La formule est numériquement instable pour les triangles
  très petits (arccos près de $1$) — sans incidence ici ($L/R \ge 0{,}14$), mais une version
  vectorielle robuste serait requise pour de petits triangles.

## Exercice 1 — L'antipode

Calculer la distance grand cercle de Paris à son antipode (le point diamétralement opposé).
Le résultat doit être la moitié de la circonférence — le maximum possible de la distance
sphérique.

> **Indice :** l'antipode de (lat, lon) est (-lat, lon +/- 180) ; injectez-le dans gc_deg et
> comparez a pi * R.

In [8]:
# Exercice 1 -- a completer
# antipode = (-48.8566, 2.3522 - 180) ; attendu : pi * R = 20015.09 km
resultat = None  # TODO etudiant

print("Exercice 1 a completer -- attendu : distance = pi R = 20015.09 km")

Exercice 1 a completer -- attendu : distance = pi R = 20015.09 km


## Exercice 2 — Les deux caps de Paris à Tokyo

La géodésique et la loxodromie quittent Paris dans des directions très différentes. Calculer
le **cap initial** de chaque route (l'angle depuis le nord, dans $[0°, 360°)$) : la formule
sphérique $\alpha = \operatorname{atan2}(\sin\Delta\lambda\cos\varphi_2,\;
\cos\varphi_1\sin\varphi_2 - \sin\varphi_1\cos\varphi_2\cos\Delta\lambda)$ pour la
géodésique, $\operatorname{atan2}(\Delta\lambda, \Delta\psi)$ pour la loxodromie.

> **Indice :** attendu ~33 deg (nord-est) pour la geodesique contre ~97 deg (est-sud-est) pour la
> loxodromie -- soixante-quatre degres d'ecart au depart, pour se rejoindre a Tokyo.

In [9]:
# Exercice 2 -- a completer
# attendu : cap geodesique ~ 33.5 deg, cap loxodromique ~ 97.4 deg
resultat = None  # TODO etudiant

print("Exercice 2 a completer -- attendu : 33.5 deg vs 97.4 deg, ecart 64 deg")

Exercice 2 a completer -- attendu : 33.5 deg vs 97.4 deg, ecart 64 deg


## Exercice 3 — L'octant : trois angles droits

Construire le triangle formé du pôle Nord, du point $(0°, 0°)$ et du point $(0°, 90°E)$ :
trois arcs de grands cercles qui se coupent à angle droit à chaque sommet. Vérifier les trois
angles $= 90°$ et l'aire par Gauss-Bonnet — un huitième de la sphère.

> **Indice :** trois angles droits donnent un exces de pi/2 ; l'aire attendue est (pi/2) R^2 =
> 4 pi R^2 / 8 = 63758059 km^2.

In [10]:
# Exercice 3 -- a completer
# sommets : pole Nord, (0, 0), (0, 90) ; attendu : 3 angles droits, aire = (pi/2) R^2
resultat = None  # TODO etudiant

print("Exercice 3 a completer -- attendu : aire = pi/2 R^2 = 63758059 km^2, soit 12.5 % de la sphere")

Exercice 3 a completer -- attendu : aire = pi/2 R^2 = 63758059 km^2, soit 12.5 % de la sphere


## Références

- B. Riemann, *Über die Hypothesen, welche der Geometrie zu Grunde liegen* (leçon
  d'habilitation, Göttingen, 10 juin 1854) — la métrique comme fondement de la géométrie.
- C. F. Gauss, *Disquisitiones generales circa superficies curvas* (1827) — le
  *theorema egregium* et la *curvatura integra*.
- S. L'Huilier, *Mémoire sur le minimum de cire des alvéoles des abeilles* (1798) — la formule
  de l'aire du triangle sphérique depuis ses côtés.
- M. P. do Carmo, *Differential Geometry of Curves and Surfaces* (1976), chap. 4 — Gauss-Bonnet.
- Mathlib, `Mathlib/Geometry/Manifold/` et `Mathlib/Geometry/Euclidean/Sphere/` — le cadre
  disponible ; la courbure riemannienne reste à formaliser.
- Série Riemann200, grains R-01 à R-09 — le fil de la série
  (EPIC [#20403](https://github.com/jsboige/CoursIA/issues/20403))